# Two precursor-pattern diagnostics: main and supplementary figures

## tl;dr

This notebook reorganizes the original composite plotting script into one main figure and two independent supplementary figures.

- **Main figure:** record sections for Pattern I and Pattern II, four primary discriminating features, hourly occurrence evolution, and median PSD profiles.
- **Supplementary figure:** rainfall–precursor lag correlations and six additional waveform/energy feature distributions.

The colored vertical strips beside the record sections are removed. Every feature subplot has a y-axis label, including both lower subplots in the main feature panel. Font sizes, figure dimensions, and subplot spacing are grouped at the beginning of each plotting cell for convenient editing.

## Context & Methods

The analysis contains 35 PCS-selected states separated into two data-driven patterns. The notebook preserves the waveform selection, hourly aggregation, Welch PSD calculation and rainfall-lag calculation used by the source script. The additional supplementary boxplots are descriptive comparisons and are not used to define the two patterns.

### Figure contract

- The main figure supports the distinction between the two precursor-pattern families.
- The supplementary figure provides hydrometeorological context and additional waveform diagnostics.
- Pattern colors are used for curves, points and box outlines; record-section waveforms remain black to preserve waveform morphology.

## Setup

### Imports, paths and shared constants

In [ ]:
from pathlib import Path
import sys
import textwrap

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from obspy import read
from scipy.signal import welch
from IPython.display import display


def resolve_paper_directory() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd() / "uilt_oneday_all" / "uilt_paper_figure",
        Path.cwd().parent,
    ]
    for candidate in candidates:
        if (candidate / "05_new_ record-section.py").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate 05_new_ record-section.py")


PAPER_DIR = resolve_paper_directory()
PROJECT_ROOT = PAPER_DIR.parents[1]
OUT_DIR = PAPER_DIR / "Fig5_record_section_two_precursor_patterns" / "notebook_exports"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RESULT_DIR = next(
    PROJECT_ROOT.glob(
        "uilt_oneday_all/03visual/**/kmeans_clustering_from_matched_precursor_features"
    )
)
RESULT_CSV = RESULT_DIR / "kmeans_clustering_results.csv"
EVIDENCE_XLSX = (
    RESULT_DIR
    / "kmeans2_mode_waveform_process_evidence"
    / "kmeans2_mode_waveform_process_evidence.xlsx"
)
NPZ_DIR = (
    PROJECT_ROOT
    / "uilt_oneday_all"
    / "02clustering"
    / "result"
    / "DEC_result"
    / "all_parement"
)
SEGMENT_ROOT = PROJECT_ROOT / "uilt_oneday_all" / "01data_preprocess"

if str(PAPER_DIR) not in sys.path:
    sys.path.insert(0, str(PAPER_DIR))
from figure_rainfall_couple_event import (
    LAGS,
    build_hourly_matrices,
    lag_correlations,
    load_data as load_coupling_data,
)

MODE_NAMES = {0: "Pattern I", 1: "Pattern II"}
NYQUIST_HZ = 50.0
RANDOM_SEED = 20260701

print(f"Source table: {RESULT_CSV}")
print(f"Evidence workbook: {EVIDENCE_XLSX}")
print(f"Figure output: {OUT_DIR}")


## Data

### Load clustering results and supporting tables

In [ ]:
results = pd.read_csv(RESULT_CSV)
selected_feature_table = pd.read_excel(EVIDENCE_XLSX, sheet_name="selected_features")
hourly_count_table = pd.read_excel(EVIDENCE_XLSX, sheet_name="hourly_counts")

modes = sorted(results["kmeans_cluster"].dropna().astype(int).unique().tolist())
mode_counts = (
    results.groupby("kmeans_cluster").size().astype(int).to_dict()
)

required_columns = {
    "event", "cluster_id", "kmeans_cluster",
    "NBRPEAKFREQMAX", "DISTQ2Q1", "NBRPEAKFREQCENTER", "NBRPEAKMEAN",
    "Peak_Amplitude", "RMS", "IQR", "Relative_Energy",
    "Energy_Density", "Energy_4_12Hz", "rectilinearity",
}
missing_columns = sorted(required_columns.difference(results.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")
if modes != [0, 1]:
    raise ValueError(f"Expected two patterns labelled 0 and 1, found {modes}")
if int(results.shape[0]) != 35:
    raise ValueError(f"Expected 35 PCS-selected states, found {results.shape[0]}")

pd.DataFrame({
    "pattern": [MODE_NAMES[mode] for mode in modes],
    "n_states": [mode_counts[mode] for mode in modes],
})

### Waveform, hourly-count and spectral helpers

In [ ]:
def npz_path(event: str) -> Path:
    for name in [f"{event}_all_parement.npz", f"{event}_all_parameter.npz"]:
        path = NPZ_DIR / name
        if path.exists():
            return path
    candidates = list(NPZ_DIR.glob(f"{event}*.npz"))
    if not candidates:
        raise FileNotFoundError(f"No DEC output found for {event}")
    return candidates[0]


def resolve_sac_path(x_file) -> Path:
    backslash = chr(92)
    normalized = str(x_file).replace(backslash, "/")
    path = Path(normalized)
    if path.is_absolute() and path.exists():
        return path
    return SEGMENT_ROOT / normalized.replace("./", "").replace("." + backslash, "")


def normalize_waveform(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    values = values - np.nanmedian(values)
    scale = np.nanpercentile(np.abs(values), 99)
    return values / scale if np.isfinite(scale) and scale > 0 else values


def center_waveform(row) -> dict:
    event = str(row.event)
    cluster_id = int(row.cluster_id)
    with np.load(npz_path(event), allow_pickle=True) as archive:
        labels = np.asarray(archive["y_pred_last"]).astype(int)
        distances = np.asarray(archive["distances"], dtype=float)
        indices = np.where(labels == cluster_id)[0]
        center_index = indices[np.argmin(distances[indices, cluster_id])]
        waveform_path = resolve_sac_path(archive["x_file"][center_index])
    trace = read(str(waveform_path))[0]
    raw = np.asarray(trace.data, dtype=float)
    raw = raw - np.nanmedian(raw)
    return {
        "event": event.replace("event", "E"),
        "cluster": f"C{cluster_id}",
        "time_before_h": -24 + center_index / 60,
        "time": np.arange(trace.stats.npts) / float(trace.stats.sampling_rate),
        "waveform": normalize_waveform(raw),
        "raw": raw,
        "sampling_rate": float(trace.stats.sampling_rate),
    }


def build_hourly_mode_curves() -> dict:
    hour_columns = [f"hour_{hour}_to_{hour + 1}" for hour in range(-24, 0)]
    curves = {}
    for mode in modes:
        event_ids = set(
            results.loc[results["kmeans_cluster"].eq(mode), "event"].astype(str)
        )
        matrix = hourly_count_table.loc[
            hourly_count_table["event"].astype(str).isin(event_ids), hour_columns
        ].apply(pd.to_numeric, errors="coerce").fillna(0).to_numpy(float)
        curves[mode] = {
            "x": np.array([hour + 0.5 for hour in range(-24, 0)], dtype=float),
            "mean": np.nanmean(matrix, axis=0),
            "q25": np.nanpercentile(matrix, 25, axis=0),
            "q75": np.nanpercentile(matrix, 75, axis=0),
            "n": matrix.shape[0],
        }
    return curves


def build_psd_profiles(center_waveforms: dict) -> dict:
    profiles = {}
    for mode, items in center_waveforms.items():
        spectra = []
        frequency_out = None
        for item in items:
            waveform = np.asarray(item["raw"], dtype=float)
            sampling_rate = float(item["sampling_rate"])
            nperseg = min(waveform.size, int(sampling_rate * 8))
            frequency, psd = welch(
                waveform,
                fs=sampling_rate,
                window="hann",
                nperseg=nperseg,
                noverlap=nperseg // 2,
                detrend="constant",
                scaling="density",
            )
            keep = (frequency >= 0) & (frequency <= NYQUIST_HZ)
            frequency_out = frequency[keep]
            spectra.append(10 * np.log10(psd[keep] + np.finfo(float).eps))
        matrix = np.asarray(spectra, dtype=float)
        profiles[mode] = {
            "freq": frequency_out,
            "median": np.nanmedian(matrix, axis=0),
            "q25": np.nanpercentile(matrix, 25, axis=0),
            "q75": np.nanpercentile(matrix, 75, axis=0),
            "n": matrix.shape[0],
        }
    return profiles

### Build the reusable analysis objects

In [ ]:
waveforms = {
    mode: [
        center_waveform(row)
        for row in results.loc[results["kmeans_cluster"].eq(mode)]
            .sort_values("event")
            .itertuples(index=False)
    ]
    for mode in modes
}
hourly_curves = build_hourly_mode_curves()
psd_profiles = build_psd_profiles(waveforms)

mode_table, matched_samples, rain_table = load_coupling_data()
rainfall_matrix, precursor_matrix, rainfall_event_labels, rainfall_mode_labels = (
    build_hourly_matrices(mode_table, matched_samples, rain_table)
)
rainfall_lag_correlations = lag_correlations(rainfall_matrix, precursor_matrix)

MAIN_FEATURES = [
    "NBRPEAKFREQMAX",
    "DISTQ2Q1",
    "NBRPEAKFREQCENTER",
    "NBRPEAKMEAN",
]
SUPPLEMENTARY_FEATURES = [
    "Peak_Amplitude",
    "IQR",
    "Energy_4_12Hz",
    "rectilinearity",
]

FEATURE_TITLES = {
    "NBRPEAKFREQMAX": "Peak-frequency maximum",
    "DISTQ2Q1": "Q2-Q1 frequency distance",
    "NBRPEAKFREQCENTER": "Peak-frequency center",
    "NBRPEAKMEAN": "Peak-count mean",
    "Peak_Amplitude": "Maximum amplitude",
    "RMS": "RMS amplitude",
    "IQR": "Amplitude interquartile range",
    "Energy_4_12Hz": "35–45 Hz log-envelope energy",
    "Relative_Energy": "Relative energy",
    "Energy_Density": "Energy density",
    "rectilinearity": "Rectilinearity",
}
FEATURE_YLABELS = {
    "NBRPEAKFREQMAX": "Frequency (Hz)",
    "DISTQ2Q1": "Frequency distance (Hz)",
    "NBRPEAKFREQCENTER": "Frequency (Hz)",
    "NBRPEAKMEAN": "Peak count",
    "Peak_Amplitude": "Amplitude (a.u.)",
    "RMS": "Amplitude (a.u.)",
    "IQR": "Amplitude (a.u.)",
    "Energy_4_12Hz": "log10 envelope integral",
    "Relative_Energy": "Relative energy (a.u.)",
    "Energy_Density": "Energy density (a.u.)",
    "rectilinearity": "Rectilinearity",
}

print("Waveforms per pattern:", {mode: len(items) for mode, items in waveforms.items()})
print("Main features:", MAIN_FEATURES)
print("Supplementary features:", SUPPLEMENTARY_FEATURES)

### Prepare plot-ready and export tables

Compute summaries and feature groups once before any figure cell.

In [ ]:
# Plot-ready groups and statistical summaries used by the figure cells
main_feature_values = {
    feature: {
        mode: pd.to_numeric(
            results.loc[results["kmeans_cluster"].eq(mode), feature], errors="coerce"
        ).dropna().to_numpy(float)
        for mode in modes
    }
    for feature in MAIN_FEATURES
}
supplementary_feature_values = {
    feature: {
        mode: pd.to_numeric(
            results.loc[results["kmeans_cluster"].eq(mode), feature], errors="coerce"
        ).dropna().to_numpy(float)
        for mode in modes
    }
    for feature in SUPPLEMENTARY_FEATURES
}

rainfall_curve_summaries = {}
for mode, mode_name in zip(modes, ["Pattern I", "Pattern II"]):
    matrix = rainfall_lag_correlations[rainfall_mode_labels == mode_name]
    q25, q75 = np.nanpercentile(matrix, [25, 75], axis=0)
    rainfall_curve_summaries[mode] = {
        "median": np.nanmedian(matrix, axis=0),
        "q25": q25,
        "q75": q75,
        "n": len(matrix),
    }

effect_rows = []
for feature in SUPPLEMENTARY_FEATURES:
    group_i = supplementary_feature_values[feature][0]
    group_ii = supplementary_feature_values[feature][1]
    effect_rows.append({
        "feature": feature,
        "cliffs_delta_I_minus_II": float(
            np.mean(np.sign(group_i[:, None] - group_ii[None, :]))
        ),
    })
feature_effect_sizes = pd.DataFrame(effect_rows)

# Source-data tables are prepared before plotting; the later export cell only writes them.
feature_source_columns = [
    "event", "cluster_id", "kmeans_cluster", "kmeans_cluster_name",
    *MAIN_FEATURES, *SUPPLEMENTARY_FEATURES,
]
feature_source = results[feature_source_columns].copy()

hourly_source_rows = []
for mode in modes:
    curve = hourly_curves[mode]
    for hour, mean, q25, q75 in zip(curve["x"], curve["mean"], curve["q25"], curve["q75"]):
        hourly_source_rows.append({
            "pattern": MODE_NAMES[mode],
            "time_before_initiation_h": hour,
            "mean_hourly_count": mean,
            "q25_hourly_count": q25,
            "q75_hourly_count": q75,
        })
hourly_source = pd.DataFrame(hourly_source_rows)

psd_source_rows = []
for mode in modes:
    profile = psd_profiles[mode]
    for frequency, median, q25, q75 in zip(profile["freq"], profile["median"], profile["q25"], profile["q75"]):
        psd_source_rows.append({
            "pattern": MODE_NAMES[mode],
            "frequency_hz": frequency,
            "median_psd_db": median,
            "q25_psd_db": q25,
            "q75_psd_db": q75,
        })
psd_source = pd.DataFrame(psd_source_rows)

rainfall_source_rows = []
for mode in modes:
    summary = rainfall_curve_summaries[mode]
    for lag, center, low, high in zip(LAGS, summary["median"], summary["q25"], summary["q75"]):
        rainfall_source_rows.append({
            "pattern": MODE_NAMES[mode],
            "rainfall_lead_h": lag,
            "median_spearman_r": center,
            "q25_spearman_r": low,
            "q75_spearman_r": high,
        })
rainfall_source = pd.DataFrame(rainfall_source_rows)


## Results

### Main figure

In [ ]:
# Plot settings
figsize_mm = (183, 135); mm_per_inch = 25.4; png_dpi = 400; tiff_dpi = 600
font_add = 2
box_title_fs = 7.5
font_family = "Arial"; base_fs = 7.0 + font_add; label_fs = 7.4 + font_add; title_fs = 7.8 + font_add
tick_fs = 6.3 + font_add; legend_fs = 6.0 + font_add; panel_fs = 11.0 + font_add
feature_ylabel_fs = 7.5
mode_colors = {0: "tab:blue", 1: "#E34040"}
trace_color = "#141010"; baseline_color = "#DEDEDE"; white = "#FFFFFF"; panel_color = "#000000"
axis_lw = 0.75; tick_width = 0.60; tick_length = 2.2; fill_lw = 0

left = 0.075; right = 0.985; bottom = 0.080; top = 0.965
outer_wspace = 0.20; outer_hspace = 0.27
record_wspace = 0.27; box_wspace = 0.32; box_hspace = 0.32
outer_width_ratios = (0.49, 0.53); outer_height_ratios = (1.0, 1.0)

trace_time_limit = (0, 60); trace_scale = 0.28; trace_lw = 0.10
baseline_lw = 0.24; baseline_zorder = 0
trace_ylim_pad = (-0.8, -0.2); title_pad = 2.2

box_positions = (0, 1); box_width = 0.48; box_alpha = 0.16
box_lw = 0.75; median_lw = 1.1
scatter_jitter = 0.035; scatter_size = 12; scatter_edge_lw = 0.25
scatter_alpha = 0.78; scatter_zorder = 3
feature_xlim = (-0.55, 1.55); feature_labelpad = 2; sci_limits = (-2, 2)

hourly_lw = 1.6; hourly_marker = "o"; hourly_marker_size = 2.6
hourly_fill_alpha = 0.15
hourly_xlim = (-24, 0); hourly_xticks = [-24, -18, -12, -6, 0]
hourly_minor_ticks = np.arange(-23.5, 0, 1)
minor_tick_length = 2.0; minor_tick_width = 0.55

psd_xlim = (0, NYQUIST_HZ); psd_lw = 1.7
psd_fill_alpha = 0.15; psd_fill_zorder = 1; psd_line_zorder = 2

legend_loc = "upper left"; legend_handlelength = 1.2
panel_x_offset = 0.030; panel_y_offset = 0.018

save_bbox = "tight"; save_facecolor = "white"; save_pad = 0.04
tiff_compression = "tiff_lzw"
main_stem = OUT_DIR / "figure5_two_precursor_patterns_main"

mpl.rcParams.update({
    "font.family": font_family,
    "font.size": base_fs,
    "axes.linewidth": axis_lw,
    "axes.labelsize": label_fs,
    "axes.titlesize": title_fs,
    "xtick.labelsize": tick_fs,
    "ytick.labelsize": tick_fs,
    "legend.fontsize": legend_fs,
    "legend.frameon": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
plt.ioff()

# Plot
fig_main = plt.figure(
    figsize=(
        figsize_mm[0] / mm_per_inch,
        figsize_mm[1] / mm_per_inch,
    ),
    facecolor=save_facecolor,
)

outer = fig_main.add_gridspec(
    2,
    2,
    width_ratios=outer_width_ratios,
    height_ratios=outer_height_ratios,
    left=left,
    right=right,
    bottom=bottom,
    top=top,
    wspace=outer_wspace,
    hspace=outer_hspace,
)

# a：波形，左上
record_grid = outer[0, 0].subgridspec(
    1,
    2,
    wspace=record_wspace,
)

# c：四个特征箱线图，左下
box_grid = outer[1, 0].subgridspec(
    2,
    2,
    wspace=box_wspace,
    hspace=box_hspace,
)

ax_wave_i = fig_main.add_subplot(record_grid[0, 0])
ax_wave_ii = fig_main.add_subplot(record_grid[0, 1])

main_feature_axes = [
    fig_main.add_subplot(box_grid[row, column])
    for row in range(2)
    for column in range(2)
]

# b：小时发生次数，右上
ax_hourly = fig_main.add_subplot(outer[0, 1])

# d：PSD，右下
ax_psd = fig_main.add_subplot(outer[1, 1])

# a. Waveform record sections
for axis, mode, show_ylabel in [
    (ax_wave_i, 0, True),
    (ax_wave_ii, 1, False),
]:
    waveform_items = sorted(
        waveforms[mode],
        key=lambda item: item["time_before_h"],
    )

    for trace_index, item in enumerate(waveform_items):
        time_mask = item["time"] <= trace_time_limit[1]

        axis.plot(
            item["time"][time_mask],
            item["waveform"][time_mask] * trace_scale + trace_index,
            color=trace_color,
            linewidth=trace_lw,
        )

        axis.axhline(
            trace_index,
            color=baseline_color,
            linewidth=baseline_lw,
            zorder=baseline_zorder,
        )

    axis.set_title(
        f"{MODE_NAMES[mode]} (n={mode_counts[mode]})",
        color=mode_colors[mode],
        pad=title_pad,
    )

    axis.set_xlim(*trace_time_limit)
    axis.set_ylim(
        trace_ylim_pad[0],
        len(waveform_items) + trace_ylim_pad[1],
    )

    axis.set_xlabel("Time (s)")
    axis.set_ylabel("Trace index" if show_ylabel else "", labelpad=1)

    axis.set_yticks([0, len(waveform_items) - 1])
    axis.set_yticklabels([1, len(waveform_items)])

    axis.tick_params(axis="both", labelsize=tick_fs)
    axis.tick_params(
        axis="y",
        length=tick_length,
        width=tick_width,
    )

# c. Four principal feature distributions
main_rng = np.random.default_rng(RANDOM_SEED)

for feature_index, (axis, feature) in enumerate(
    zip(main_feature_axes, MAIN_FEATURES)
):
    for mode, position in zip(modes, box_positions):
        values = main_feature_values[feature][mode]

        axis.boxplot(
            values,
            positions=[position],
            widths=box_width,
            patch_artist=True,
            showfliers=False,
            boxprops={
                "facecolor": mode_colors[mode],
                "alpha": box_alpha,
                "edgecolor": mode_colors[mode],
                "lw": box_lw,
            },
            medianprops={
                "color": mode_colors[mode],
                "lw": median_lw,
            },
            whiskerprops={
                "color": mode_colors[mode],
                "lw": box_lw,
            },
            capprops={
                "color": mode_colors[mode],
                "lw": box_lw,
            },
        )

        axis.scatter(
            np.full(values.size, position)
            + main_rng.normal(
                0,
                scatter_jitter,
                values.size,
            ),
            values,
            s=scatter_size,
            color=mode_colors[mode],
            edgecolor=white,
            linewidth=scatter_edge_lw,
            alpha=scatter_alpha,
            zorder=scatter_zorder,
        )

    axis.set_title(
        FEATURE_TITLES[feature], fontsize=box_title_fs,
        pad=title_pad,
    )

    axis.set_ylabel(
        FEATURE_YLABELS[feature],
        labelpad=feature_labelpad,
        fontsize=feature_ylabel_fs,
    )

    axis.set_xticks(box_positions)
    axis.set_xticklabels(
        ["I", "II"] if feature_index >= 2 else []
    )

    axis.set_xlim(*feature_xlim)

    axis.ticklabel_format(
        axis="y",
        style="sci",
        scilimits=sci_limits,
    )

    axis.tick_params(
        axis="both",
        labelsize=tick_fs,
    )

    if feature_index >= 2:
        axis.set_xlabel("Pattern")

# b. Hourly occurrence evolution
for mode in modes:
    curve = hourly_curves[mode]
    color = mode_colors[mode]

    ax_hourly.plot(
        curve["x"],
        curve["mean"],
        color=color,
        marker=hourly_marker,
        markersize=hourly_marker_size,
        linewidth=hourly_lw,
        label=f"{MODE_NAMES[mode]} (n={curve['n']})",
    )

    ax_hourly.fill_between(
        curve["x"],
        curve["q25"],
        curve["q75"],
        color=color,
        alpha=hourly_fill_alpha,
        linewidth=fill_lw,
    )

# d. Median PSD with interquartile shading
for mode in modes:
    profile = psd_profiles[mode]
    color = mode_colors[mode]

    ax_psd.fill_between(
        profile["freq"],
        profile["q25"],
        profile["q75"],
        color=color,
        alpha=psd_fill_alpha,
        linewidth=fill_lw,
        zorder=psd_fill_zorder,
    )

    ax_psd.plot(
        profile["freq"],
        profile["median"],
        color=color,
        linewidth=psd_lw,
        label=f"{MODE_NAMES[mode]} (n={profile['n']})",
        zorder=psd_line_zorder,
    )

# Axis and layout
ax_hourly.set_xlim(*hourly_xlim)
ax_hourly.set_xticks(hourly_xticks)
ax_hourly.set_xticks(hourly_minor_ticks, minor=True)

ax_hourly.tick_params(
    axis="both",
    labelsize=tick_fs,
)

ax_hourly.tick_params(
    axis="x",
    which="minor",
    length=minor_tick_length,
    width=minor_tick_width,
)

ax_hourly.set_xlabel(
    "Time before debris-flow initiation (h)"
)

ax_hourly.set_ylabel(
    "Hourly occurrence count"
)

ax_hourly.legend(
    loc=legend_loc,
    handlelength=legend_handlelength,
)

ax_psd.set_xlim(*psd_xlim)

ax_psd.tick_params(
    axis="both",
    labelsize=tick_fs,
)

ax_psd.set_xlabel("Frequency (Hz)")
ax_psd.set_ylabel("PSD (dB)")

ax_psd.legend(
    loc=legend_loc,
    handlelength=legend_handlelength,
)

# Panel labels：a b / c d
panel_specs = {
    "a": outer[0, 0],
    "b": outer[0, 1],
    "c": outer[1, 0],
    "d": outer[1, 1],
}

for label, subplot_spec in panel_specs.items():
    panel_bbox = subplot_spec.get_position(fig_main)

    fig_main.text(
        panel_bbox.x0 - panel_x_offset,
        panel_bbox.y1 + panel_y_offset,
        label,
        transform=fig_main.transFigure,
        fontsize=panel_fs,
        fontweight="bold",
        ha="left",
        va="top",
        color=panel_color,
    )
for ax in [ax_wave_i,ax_wave_ii,*main_feature_axes,ax_hourly,ax_psd]:
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(axis_lw)
    ax.tick_params(axis="both",which="both",direction="out",bottom=True,left=True,top=False,right=False,labeltop=False,labelright=False)
    ax.tick_params(axis="both",which="major",length=tick_length,width=tick_width)
    
from pathlib import Path
save_root=Path(r"D:\Pycharm_Project\Find_precursor_signs\ILLgarben\uilt_oneday_all\uilt_paper_figure\Fig_all")
(save_root/"png").mkdir(parents=True,exist_ok=True)
(save_root/"tiff").mkdir(parents=True,exist_ok=True)
png_path=save_root/"png"/"Fig 7.png"
tiff_path=save_root/"tiff"/"Fig 7.tiff"
fig_main.savefig(png_path,format="png",dpi=png_dpi,bbox_inches="tight",facecolor="white",transparent=False)
fig_main.savefig(tiff_path,format="tiff",dpi=tiff_dpi,bbox_inches="tight",facecolor="white",transparent=False,pil_kwargs={"compression":tiff_compression})
display(fig_main)
plt.close(fig_main)
print("Saved Fig 7:")
print(png_path)
print(tiff_path)
# Save
OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

fig_main.savefig(
    main_stem.with_suffix(".png"),
    dpi=png_dpi,
    bbox_inches=save_bbox,
    facecolor=save_facecolor,
    pad_inches=save_pad,
)

fig_main.savefig(
    main_stem.with_suffix(".tiff"),
    dpi=tiff_dpi,
    bbox_inches=save_bbox,
    facecolor=save_facecolor,
    pad_inches=save_pad,
    pil_kwargs={
        "compression": tiff_compression,
    },
)

display(fig_main)
plt.close(fig_main)
print(OUT_DIR)

In [ ]:
# Plot settings
figsize_mm = (183, 135); mm_per_inch = 25.4; png_dpi = 400; tiff_dpi = 600
font_add = 2; box_title_fs = 7.5
font_family = "Arial"; base_fs = 7.0 + font_add; label_fs = 7.4 + font_add; title_fs = 7.8 + font_add
tick_fs = 6.3 + font_add; legend_fs = 6.0 + font_add; panel_fs = 11.0 + font_add
feature_ylabel_fs = 7.5
mode_colors = {0: "tab:blue", 1: "#E34040"}
trace_color = "#141010"; baseline_color = "#DEDEDE"; white = "#FFFFFF"; panel_color = "#000000"
axis_lw = 0.75; tick_width = 0.60; tick_length = 2.2; fill_lw = 0

left = 0.075; right = 0.985; bottom = 0.080; top = 0.965
outer_wspace = 0.20; outer_hspace = 0.27
record_wspace = 0.27; box_wspace = 0.47; box_hspace = 0.62
outer_width_ratios = (0.49, 0.53); outer_height_ratios = (1.0, 1.0)

trace_time_limit = (0, 60); trace_scale = 0.28; trace_lw = 0.10
baseline_lw = 0.24; baseline_zorder = 0
trace_ylim_pad = (-0.8, -0.2); title_pad = 2.2

box_positions = (0, 1); box_width = 0.48; box_alpha = 0.16
box_lw = 0.75; median_lw = 1.1
scatter_jitter = 0.035; scatter_size = 12; scatter_edge_lw = 0.25
scatter_alpha = 0.78; scatter_zorder = 3
feature_xlim = (-0.55, 1.55); feature_labelpad = 2; sci_limits = (-2, 2)

hourly_lw = 1.6; hourly_marker = "o"; hourly_marker_size = 2.6
hourly_fill_alpha = 0.15
hourly_xlim = (-24, 0); hourly_xticks = [-24, -18, -12, -6, 0]
hourly_minor_ticks = np.arange(-23.5, 0, 1)
minor_tick_length = 2.0; minor_tick_width = 0.55

psd_xlim = (0, NYQUIST_HZ); psd_lw = 1.7
psd_fill_alpha = 0.15; psd_fill_zorder = 1; psd_line_zorder = 2

legend_loc = "upper left"; legend_handlelength = 1.2

# Independent panel-label offsets: (x, y)
panel_a_offset = (-0.030, 0.04)
panel_b_offset = (-0.030, 0.04)
panel_c_offset = (-0.030, 0.05)
panel_d_offset = (-0.030, 0.05)

save_bbox = "tight"; save_facecolor = "white"; save_pad = 0.04
tiff_compression = "tiff_lzw"
main_stem = OUT_DIR / "figure5_two_precursor_patterns_main"

mpl.rcParams.update({
    "font.family": font_family,
    "font.size": base_fs,
    "axes.linewidth": axis_lw,
    "axes.labelsize": label_fs,
    "axes.titlesize": title_fs,
    "xtick.labelsize": tick_fs,
    "ytick.labelsize": tick_fs,
    "legend.fontsize": legend_fs,
    "legend.frameon": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
plt.ioff()

# Plot
fig_main = plt.figure(
    figsize=(
        figsize_mm[0] / mm_per_inch,
        figsize_mm[1] / mm_per_inch,
    ),
    facecolor=save_facecolor,
)

outer = fig_main.add_gridspec(
    2,
    2,
    width_ratios=outer_width_ratios,
    height_ratios=outer_height_ratios,
    left=left,
    right=right,
    bottom=bottom,
    top=top,
    wspace=outer_wspace,
    hspace=outer_hspace,
)

# a：波形，左上
record_grid = outer[0, 0].subgridspec(
    1,
    2,
    wspace=record_wspace,
)

# c：四个特征箱线图，左下
box_grid = outer[1, 0].subgridspec(
    2,
    2,
    wspace=box_wspace,
    hspace=box_hspace,
)

ax_wave_i = fig_main.add_subplot(record_grid[0, 0])
ax_wave_ii = fig_main.add_subplot(record_grid[0, 1])

main_feature_axes = [
    fig_main.add_subplot(box_grid[row, column])
    for row in range(2)
    for column in range(2)
]

# b：小时发生次数，右上
ax_hourly = fig_main.add_subplot(outer[0, 1])

# d：PSD，右下
ax_psd = fig_main.add_subplot(outer[1, 1])

# a. Waveform record sections
for axis, mode, show_ylabel in [
    (ax_wave_i, 0, True),
    (ax_wave_ii, 1, False),
]:
    waveform_items = sorted(
        waveforms[mode],
        key=lambda item: item["time_before_h"],
    )

    for trace_index, item in enumerate(waveform_items):
        time_mask = item["time"] <= trace_time_limit[1]

        axis.plot(
            item["time"][time_mask],
            item["waveform"][time_mask] * trace_scale + trace_index,
            color=trace_color,
            linewidth=trace_lw,
        )

        axis.axhline(
            trace_index,
            color=baseline_color,
            linewidth=baseline_lw,
            zorder=baseline_zorder,
        )

    axis.set_title(
        f"{MODE_NAMES[mode]} (n={mode_counts[mode]})",
        color=mode_colors[mode],
        pad=title_pad,
    )

    axis.set_xlim(*trace_time_limit)
    axis.set_ylim(
        trace_ylim_pad[0],
        len(waveform_items) + trace_ylim_pad[1],
    )

    axis.set_xlabel("Time (s)")
    axis.set_ylabel("Trace index" if show_ylabel else "", labelpad=1)

    axis.set_yticks([0, len(waveform_items) - 1])
    axis.set_yticklabels([1, len(waveform_items)])

    axis.tick_params(axis="both", labelsize=tick_fs)

    axis.tick_params(
        axis="y",
        length=tick_length,
        width=tick_width,
    )

# c. Four principal feature distributions
main_rng = np.random.default_rng(RANDOM_SEED)

for feature_index, (axis, feature) in enumerate(
    zip(main_feature_axes, MAIN_FEATURES)
):
    for mode, position in zip(modes, box_positions):
        values = main_feature_values[feature][mode]

        axis.boxplot(
            values,
            positions=[position],
            widths=box_width,
            patch_artist=True,
            showfliers=False,
            boxprops={
                "facecolor": mode_colors[mode],
                "alpha": box_alpha,
                "edgecolor": mode_colors[mode],
                "lw": box_lw,
            },
            medianprops={
                "color": mode_colors[mode],
                "lw": median_lw,
            },
            whiskerprops={
                "color": mode_colors[mode],
                "lw": box_lw,
            },
            capprops={
                "color": mode_colors[mode],
                "lw": box_lw,
            },
        )

        axis.scatter(
            np.full(values.size, position)
            + main_rng.normal(
                0,
                scatter_jitter,
                values.size,
            ),
            values,
            s=scatter_size,
            color=mode_colors[mode],
            edgecolor=white,
            linewidth=scatter_edge_lw,
            alpha=scatter_alpha,
            zorder=scatter_zorder,
        )

    axis.set_title(
        FEATURE_TITLES[feature],
        fontsize=box_title_fs,
        pad=title_pad,
    )

    axis.set_ylabel(
        FEATURE_YLABELS[feature],
        labelpad=feature_labelpad,
        fontsize=feature_ylabel_fs,
    )

    axis.set_xticks(box_positions)

    axis.set_xticklabels(
        ["I", "II"] if feature_index >= 2 else []
    )

    axis.set_xlim(*feature_xlim)

    axis.ticklabel_format(
        axis="y",
        style="sci",
        scilimits=sci_limits,
    )

    axis.tick_params(
        axis="both",
        labelsize=tick_fs,
    )

    if feature_index >= 2:
        axis.set_xlabel("Pattern")

# b. Hourly occurrence evolution
for mode in modes:
    curve = hourly_curves[mode]
    color = mode_colors[mode]

    ax_hourly.plot(
        curve["x"],
        curve["mean"],
        color=color,
        marker=hourly_marker,
        markersize=hourly_marker_size,
        linewidth=hourly_lw,
        label=f"{MODE_NAMES[mode]} (n={curve['n']})",
    )

    ax_hourly.fill_between(
        curve["x"],
        curve["q25"],
        curve["q75"],
        color=color,
        alpha=hourly_fill_alpha,
        linewidth=fill_lw,
    )

# d. Median PSD with interquartile shading
for mode in modes:
    profile = psd_profiles[mode]
    color = mode_colors[mode]

    ax_psd.fill_between(
        profile["freq"],
        profile["q25"],
        profile["q75"],
        color=color,
        alpha=psd_fill_alpha,
        linewidth=fill_lw,
        zorder=psd_fill_zorder,
    )

    ax_psd.plot(
        profile["freq"],
        profile["median"],
        color=color,
        linewidth=psd_lw,
        label=f"{MODE_NAMES[mode]} (n={profile['n']})",
        zorder=psd_line_zorder,
    )

# Axis and layout
ax_hourly.set_xlim(*hourly_xlim)
ax_hourly.set_xticks(hourly_xticks)
ax_hourly.set_xticks(hourly_minor_ticks, minor=True)

ax_hourly.tick_params(
    axis="both",
    labelsize=tick_fs,
)

ax_hourly.tick_params(
    axis="x",
    which="minor",
    length=minor_tick_length,
    width=minor_tick_width,
)

ax_hourly.set_xlabel(
    "Time before debris-flow initiation (h)"
)

ax_hourly.set_ylabel(
    "Hourly occurrence count"
)

ax_hourly.legend(
    loc=legend_loc,
    handlelength=legend_handlelength,
)

ax_psd.set_xlim(*psd_xlim)

ax_psd.tick_params(
    axis="both",
    labelsize=tick_fs,
)

ax_psd.set_xlabel("Frequency (Hz)")
ax_psd.set_ylabel("PSD (dB)")

ax_psd.legend(
    loc=legend_loc,
    handlelength=legend_handlelength,
)

# Panel labels：a b / c d
panel_settings = {
    "a": (outer[0, 0], panel_a_offset),
    "b": (outer[0, 1], panel_b_offset),
    "c": (outer[1, 0], panel_c_offset),
    "d": (outer[1, 1], panel_d_offset),
}

for label, (subplot_spec, panel_offset) in panel_settings.items():
    panel_bbox = subplot_spec.get_position(fig_main)
    panel_dx, panel_dy = panel_offset

    fig_main.text(
        panel_bbox.x0 + panel_dx,
        panel_bbox.y1 + panel_dy,
        label,
        transform=fig_main.transFigure,
        fontsize=panel_fs,
        fontweight="bold",
        ha="left",
        va="top",
        color=panel_color,
    )

# Save
OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

fig_main.savefig(
    main_stem.with_suffix(".png"),
    dpi=png_dpi,
    bbox_inches=save_bbox,
    facecolor=save_facecolor,
    pad_inches=save_pad,
)

fig_main.savefig(
    main_stem.with_suffix(".tiff"),
    dpi=tiff_dpi,
    bbox_inches=save_bbox,
    facecolor=save_facecolor,
    pad_inches=save_pad,
    pil_kwargs={
        "compression": tiff_compression,
    },
)

display(fig_main)
plt.close(fig_main)
print(OUT_DIR)

### Supplementary rainfall figure

Rainfall–precursor coupling is exported separately because it tests an external hydrological relationship rather than waveform-state separation.

In [ ]:
# Plot settings
figsize_mm = (183, 62); mm_per_inch = 25.4; png_dpi = 400; tiff_dpi = 600
font_family = "Arial"; base_fs = 7.0; label_fs = 7.4; title_fs = 7.8; tick_fs = 6.3; legend_fs = 6.0
mode_colors = {0: "#1F77B4", 1: "#E76F51"}; zero_color = "#737373"
axis_lw = 0.75; tick_width = 0.70; tick_length = 3.0; curve_lw = 1.6; fill_lw = 0; zero_lw = 0.8
marker = "o"; marker_size = 3.2; fill_alpha = 0.15; zero_ls = "--"
left = 0.085; right = 0.985; bottom = 0.245; top = 0.900
xlim = (-0.3, 12.3); ylim = (-0.65, 0.65); xticks = LAGS
title_loc = "left"; title_pad = 3; legend_loc = "upper right"; legend_ncol = 2; legend_handlelength = 1.4
save_bbox = "tight"; save_facecolor = "white"; save_pad = 0.04; tiff_compression = "tiff_lzw"
rainfall_stem = OUT_DIR / "supplementary_rainfall_precursor_coupling"

mpl.rcParams.update({"font.family": font_family, "font.size": base_fs, "axes.linewidth": axis_lw,
                     "axes.labelsize": label_fs, "axes.titlesize": title_fs, "xtick.labelsize": tick_fs,
                     "ytick.labelsize": tick_fs, "legend.fontsize": legend_fs, "legend.frameon": False,
                     "axes.spines.top": False, "axes.spines.right": False})
plt.ioff()

# Plot
fig_rain, ax_rainfall = plt.subplots(figsize=(figsize_mm[0] / mm_per_inch, figsize_mm[1] / mm_per_inch), facecolor=save_facecolor)
fig_rain.subplots_adjust(left=left, right=right, bottom=bottom, top=top)
for mode in modes:
    summary = rainfall_curve_summaries[mode]; color = mode_colors[mode]
    ax_rainfall.plot(LAGS, summary["median"], color=color, marker=marker, markersize=marker_size,
                     linewidth=curve_lw, label=f"{MODE_NAMES[mode]} (n={summary['n']})")
    ax_rainfall.fill_between(LAGS, summary["q25"], summary["q75"], color=color, alpha=fill_alpha, linewidth=fill_lw)
ax_rainfall.axhline(0, color=zero_color, linestyle=zero_ls, linewidth=zero_lw)

# Axis and layout
ax_rainfall.set_xlim(*xlim); ax_rainfall.set_ylim(*ylim); ax_rainfall.set_xticks(xticks)
ax_rainfall.set_xlabel("Rainfall lead time relative to precursor activity (h)")
ax_rainfall.set_ylabel("Event-wise Spearman correlation")
ax_rainfall.set_title("Event-scale rainfall-precursor coupling", loc=title_loc, pad=title_pad)
ax_rainfall.tick_params(axis="both", labelsize=tick_fs, width=tick_width, length=tick_length)
ax_rainfall.legend(loc=legend_loc, ncol=legend_ncol, handlelength=legend_handlelength)

# Save
OUT_DIR.mkdir(parents=True, exist_ok=True)
fig_rain.savefig(rainfall_stem.with_suffix(".png"), dpi=png_dpi, bbox_inches=save_bbox, facecolor=save_facecolor, pad_inches=save_pad)
fig_rain.savefig(rainfall_stem.with_suffix(".tiff"), dpi=tiff_dpi, bbox_inches=save_bbox, facecolor=save_facecolor,
                  pad_inches=save_pad, pil_kwargs={"compression": tiff_compression})
display(fig_rain); plt.close(fig_rain)


### Supplementary waveform and polarization characteristics

Four complementary descriptors are shown in a 2 × 2 layout: maximum amplitude, amplitude interquartile range, 35–45 Hz log-envelope energy and rectilinearity. They summarize peak signal strength, sustained amplitude variability, high-frequency energy and polarization geometry, respectively. The plots are descriptive because these variables contributed to the secondary clustering.

In [ ]:
# Plot settings
figsize_mm = (150, 110); mm_per_inch = 25.4; png_dpi = 400; tiff_dpi = 600
font_family = "Arial"; base_fs = 9.0; label_fs = 9.4; title_fs = 9.8; tick_fs = 8.3; panel_fs = 12.0
white = "#FFFFFF"; panel_color = "#000000"
mode_colors = {0: "tab:blue", 1: "#E34040"}
axis_lw = 0.75; tick_width = 0.70; tick_length = 3.0
left = 0.095; right = 0.985; bottom = 0.115; top = 0.940; wspace = 0.25; hspace = 0.30
box_positions = (0, 1); box_width = 0.48; box_alpha = 0.16; box_lw = 0.75; median_lw = 1.1
scatter_jitter = 0.035; scatter_size = 12; scatter_edge_lw = 0.25; scatter_alpha = 0.78; scatter_zorder = 3; rng_offset = 20
xlim = (-0.55, 1.55); title_pad = 2; labelpad = 2; sci_limits = (-2, 2); panel_xy = (-0.20, 1.10)
save_bbox = "tight"; save_facecolor = "white"; save_pad = 0.04; tiff_compression = "tiff_lzw"
feature_stem = OUT_DIR / "supplementary_discriminating_waveform_features"

mpl.rcParams.update({"font.family": font_family, "font.size": base_fs, "axes.linewidth": axis_lw,
                     "axes.labelsize": label_fs, "axes.titlesize": title_fs, "xtick.labelsize": tick_fs,
                     "ytick.labelsize": tick_fs, "axes.spines.top": False, "axes.spines.right": False})
plt.ioff()

# Plot
fig_features, feature_axes = plt.subplots(2, 2, figsize=(figsize_mm[0] / mm_per_inch, figsize_mm[1] / mm_per_inch), facecolor=save_facecolor)
fig_features.subplots_adjust(left=left, right=right, bottom=bottom, top=top, wspace=wspace, hspace=hspace)
feature_axes = feature_axes.ravel(); feature_rng = np.random.default_rng(RANDOM_SEED + rng_offset)

for feature_index, (axis, feature) in enumerate(zip(feature_axes, SUPPLEMENTARY_FEATURES)):
    for mode, position in zip(modes, box_positions):
        values = supplementary_feature_values[feature][mode]
        axis.boxplot(
            values, positions=[position], widths=box_width, patch_artist=True, showfliers=False,
            boxprops={"facecolor": mode_colors[mode], "alpha": box_alpha, "edgecolor": mode_colors[mode], "lw": box_lw},
            medianprops={"color": mode_colors[mode], "lw": median_lw},
            whiskerprops={"color": mode_colors[mode], "lw": box_lw}, capprops={"color": mode_colors[mode], "lw": box_lw},
        )
        axis.scatter(
            np.full(values.size, position) + feature_rng.normal(0, scatter_jitter, values.size), values,
            s=scatter_size, color=mode_colors[mode], edgecolor=white, linewidth=scatter_edge_lw,
            alpha=scatter_alpha, zorder=scatter_zorder,
        )
    axis.tick_params(axis="x",pad=1.0)
    axis.set_title(FEATURE_TITLES[feature], pad=title_pad); axis.set_ylabel(FEATURE_YLABELS[feature], labelpad=labelpad)
    axis.set_xlabel("Pattern"); axis.set_xticks(box_positions); axis.set_xticklabels(["I", "II"])
    axis.set_xlim(*xlim); axis.ticklabel_format(axis="y", style="sci", scilimits=sci_limits)
    axis.tick_params(axis="both", labelsize=tick_fs, width=tick_width, length=tick_length)
    axis.text(*panel_xy, chr(ord("a") + feature_index), transform=axis.transAxes, fontsize=panel_fs,
              fontweight="bold", ha="left", va="top", color=panel_color)

# Axis and layout
for axis in feature_axes:
    axis.tick_params(axis="both", labelsize=tick_fs)
# Axis and layout
for axis in feature_axes:
    for spine in axis.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(axis_lw)
    axis.tick_params(axis="both",which="both",direction="out",labelsize=tick_fs,bottom=True,left=True,top=False,right=False,labeltop=False,labelright=False)
    axis.tick_params(axis="both",which="major",width=tick_width,length=tick_length)

# Save
from pathlib import Path
save_root=Path(r"D:\Pycharm_Project\Find_precursor_signs\ILLgarben\uilt_oneday_all\uilt_paper_figure\Fig_all")
(save_root/"png").mkdir(parents=True,exist_ok=True)
(save_root/"tiff").mkdir(parents=True,exist_ok=True)
png_path=save_root/"png"/"Fig S8.png"
tiff_path=save_root/"tiff"/"Fig S8.tiff"
fig_features.savefig(png_path,format="png",dpi=png_dpi,bbox_inches=save_bbox,facecolor=save_facecolor,pad_inches=save_pad)
fig_features.savefig(tiff_path,format="tiff",dpi=tiff_dpi,bbox_inches=save_bbox,facecolor=save_facecolor,pad_inches=save_pad,pil_kwargs={"compression":tiff_compression})
display(fig_features)
plt.close(fig_features)
print("Saved Fig S8:")
print(png_path)
print(tiff_path)
# Save
OUT_DIR.mkdir(parents=True, exist_ok=True)
fig_features.savefig(feature_stem.with_suffix(".png"), dpi=png_dpi, bbox_inches=save_bbox, facecolor=save_facecolor, pad_inches=save_pad)
fig_features.savefig(feature_stem.with_suffix(".tiff"), dpi=tiff_dpi, bbox_inches=save_bbox, facecolor=save_facecolor,
                      pad_inches=save_pad, pil_kwargs={"compression": tiff_compression})
display(fig_features); plt.close(fig_features)


### Export figure-specific source data

In [ ]:
source_data_path = OUT_DIR / "figure5_main_and_supplement_source_data.xlsx"
with pd.ExcelWriter(source_data_path, engine="openpyxl") as writer:
    feature_source.to_excel(writer, sheet_name="feature_values", index=False)
    hourly_source.to_excel(writer, sheet_name="hourly_evolution", index=False)
    psd_source.to_excel(writer, sheet_name="psd_profiles", index=False)
    rainfall_source.to_excel(writer, sheet_name="rainfall_lag", index=False)
    feature_effect_sizes.to_excel(writer, sheet_name="feature_effect_sizes", index=False)

print(source_data_path)


## Takeaways

- The main figure retains waveform morphology, the primary discriminating features, temporal evolution and PSD evidence.
- Rainfall coupling and waveform-feature separation remain independent supplementary figures because they address different parts of the evidence chain.
- The 2 × 2 supplementary feature figure compares maximum amplitude, amplitude interquartile range, 35–45 Hz log-envelope energy and rectilinearity without statistical annotations inside the panels.
- Maximum amplitude is retained as a physically intuitive descriptor, although it should not be described as one of the strongest group-separating variables.